# GD - Homework 2

In [1]:
import numpy as np
import matplotlib.pyplot as plt

## 3. Implement GD, Polyak GD, Nesterov GD, and AdaGrad GD.

* GD: xk+1 = xk−γ∇f (xk )
* Polyak: xk+1 = xk−γ∇f (xk ) + µ(xk−xk−1)
* Nesterov: xk+1 = xk−γ∇f xk + µ(xk−xk−1) + µ(xk−xk−1)
* AdaGrad: xk+1 = xk−γDk ·∇f (xk )

In [2]:
def gradient_descent(grad_f, x0, gamma, n_iters):
    x = np.array(x0, dtype=float)
    history = [x.copy()]
    for _ in range(n_iters):
        grad = grad_f(x)
        x = x - gamma * grad
        history.append(x.copy())
    return np.array(history)

def polyak_gd(grad_f, x0, gamma, mu, n_iters):
    x = np.array(x0, dtype=float)
    x_prev = x.copy()
    history = [x.copy()]
    for i in range(n_iters):
        grad = grad_f(x)
        if i == 0:
            #standard GD for first iter
            x_next = x - gamma * grad
        else:
            x_next = x - gamma * grad + mu * (x - x_prev)   
        x_prev = x.copy()
        x = x_next.copy()
        history.append(x.copy())  
    return np.array(history)

def nesterov_gd(grad_f, x0, gamma, mu, n_iters):
    x = np.array(x0, dtype=float)
    x_prev = x.copy()
    history = [x.copy()]
    for i in range(n_iters):
        if i == 0:
            #standard GD
            grad = grad_f(x)
            x_next = x - gamma * grad
        else:
            #momentum first
            x_lookahead = x + mu * (x - x_prev)
            grad = grad_f(x_lookahead)
            x_next = x - gamma * grad + mu * (x - x_prev)           
        x_prev = x.copy()
        x = x_next.copy()
        history.append(x.copy())
    return np.array(history)

def adagrad_gd(grad_f, x0, gamma, n_iters, epsilon=1e-8):
    x = np.array(x0, dtype=float)
    history = [x.copy()]
    # Cumulative sum of squared gradients for each coordinate
    sq_grad_sum = np.zeros_like(x)
    for _ in range(n_iters):
        grad = grad_f(x)
        sq_grad_sum += grad**2
        #Dk mtx
        D_k = 1.0 / (np.sqrt(sq_grad_sum) + epsilon)
        # component-wise update
        x = x - gamma * D_k * grad
        history.append(x.copy())
    return np.array(history)

**TEST:** on function $f(x,y)=x^2 + 5y^2$ with minimum at (0,0)... the same function as in the notes (fig 15,16)

**GRADIENT:** $\nabla f(x,y) = (2x, 10y)$. The Hessian has eigenvalues $\alpha = 2$ and $\beta = 10$.

**GD param:** optimal LR for a quadratic is $\gamma = \frac{2}{\alpha + \beta} = \frac{2}{12} \approx 0.166$.  

**Polyak params:** $\gamma = \frac{4}{(\sqrt{\alpha} + \sqrt{\beta})^2}$ and $\mu = \left(\frac{\sqrt{\beta} - \sqrt{\alpha}}{\sqrt{\beta} + \sqrt{\alpha}}\right)^2$. So we get $\gamma \approx 0.19$ and $\mu \approx 0.15$

**Nesterov params:** $\gamma = \frac{1}{\beta}$ and $\mu = \frac{\sqrt{\kappa} - 1}{\sqrt{\kappa} + 1}$, where $\kappa = \frac{\beta}{\alpha}$... (theorem 5.3). So this gives us $\gamma = 0.1$ and $\mu \approx 0.382$

**AdaGrad param:** standard learning rate ($\gamma = 1.0$) because AdaGrad independently weights each coordinate by accumulating past gradients, meaning it adapts its own step size.

In [9]:
def f(v):
    x, y = v
    return x**2 + 5 * y**2

def grad_f(v):
    x, y = v
    return np.array([2*x, 10*y])

#initial params
x0 = [1.0, 1.0]
n_iterations = 10

#optimal params (based on alpha=2, beta=10)
gamma_gd = 1/6
gamma_polyak = 4 / (np.sqrt(2) + np.sqrt(10))**2
mu_polyak = ((np.sqrt(10) - np.sqrt(2)) / (np.sqrt(10) + np.sqrt(2)))**2
kappa = 10 / 2
gamma_nesterov = 1 / 10
mu_nesterov = (np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1)

hist_gd = gradient_descent(grad_f, x0, gamma=gamma_gd, n_iters=n_iterations)
hist_polyak = polyak_gd(grad_f, x0, gamma=gamma_polyak, mu=mu_polyak, n_iters=n_iterations)
hist_nesterov = nesterov_gd(grad_f, x0, gamma=gamma_nesterov, mu=mu_nesterov, n_iters=n_iterations)
hist_adagrad = adagrad_gd(grad_f, x0, gamma=1.0, n_iters=n_iterations)

print("Final positions after 10 iterations (target: [0.0, 0.0]):\n")
print(f"Standard GD: {hist_gd[-1]} | f(x,y) = {f(hist_gd[-1]):.6f}")
print(f"Polyak GD:   {hist_polyak[-1]} | f(x,y) = {f(hist_polyak[-1]):.6f}")
print(f"Nesterov GD: {hist_nesterov[-1]} | f(x,y) = {f(hist_nesterov[-1]):.6f}")
print(f"AdaGrad GD:  {hist_adagrad[-1]} | f(x,y) = {f(hist_adagrad[-1]):.6f}")

Final positions after 10 iterations (target: [0.0, 0.0]):

Standard GD: [0.01734153 0.01734153] | f(x,y) = 0.001804
Polyak GD:   [0.00047467 0.00097968] | f(x,y) = 0.000005
Nesterov GD: [0.01457909 0.        ] | f(x,y) = 0.000213
AdaGrad GD:  [9.76562404e-84 1.00000069e-90] | f(x,y) = 0.000000


## 4. Find, describe, and implement Adam GD.

Adam (Adaptive Moment Estimation) combines the heavy-ball momentum from Polyak/Nesterov with the adaptive, coordinate-wise scaling of AdaGrad/RMSProp.  Instead of just adapting the learning rate based on the sum of past squared gradients (like AdaGrad, which eventually dampens the process too much ), Adam keeps an exponentially decaying average of past squared gradients. It also keeps an exponentially decaying average of past gradients (like momentum).

At each step k it calculates:
* First moment (mean): $m_k = \beta_1 m_{k-1} + (1 - \beta_1) \nabla f(x_{k-1})$
* Second moment (var): $v_k = \beta_2 v_{k-1} + (1 - \beta_2) (\nabla f(x_{k-1}))^2$

Bias correction step (to fix biasness toward 0):
* $\hat{m}_k = \frac{m_k}{1 - \beta_1^k}$
* $\hat{v}_k = \frac{v_k}{1 - \beta_2^k}$

Update rule:
$x_k = x_{k-1} - \gamma \frac{\hat{m}_k}{\sqrt{\hat{v}_k} + \epsilon}$

Params:
* $\gamma$ - global LR (npr 0.001)
* $\beta_1$ - exponential decay rate for first moment (npr 0.9)
* $\beta_2$ - exponential decay rate for second moment (npr 0.999)
* $\epsilon$ - prevent divison by 0

In [10]:
def adam_gd(grad_f, x0, gamma=0.1, n_iters=100, beta1=0.9, beta2=0.999, epsilon=1e-8):
    x = np.array(x0, dtype=float)
    history = [x.copy()]
    #first and second moments to zero
    m = np.zeros_like(x)
    v = np.zeros_like(x)
    
    for k in range(1, n_iters + 1):
        grad = grad_f(x)
        #update biased first moment
        m = beta1 * m + (1 - beta1) * grad
        #ppdate biased second raw moment
        v = beta2 * v + (1 - beta2) * (grad**2)
        #compute bias-corrected first moment estimate
        m_hat = m / (1 - beta1**k)
        #compute bias-corrected second raw moment estimate
        v_hat = v / (1 - beta2**k)
        
        # update params
        x = x - gamma * m_hat / (np.sqrt(v_hat) + epsilon)
        history.append(x.copy())
        
    return np.array(history)

In [15]:
def f(v):
    x, y = v
    return x**2 + 5 * y**2

def grad_f(v):
    x, y = v
    return np.array([2*x, 10*y])

#initial params
x0 = [1.0, 1.0]
n_iterations = 100

#use LR of 0.5 for faster convergence on this simple quadratic
hist_adam = adam_gd(grad_f, x0, gamma=0.5, n_iters=n_iterations)

print("Final position after 100 iterations (target: [0.0, 0.0]):\n")
print(f"Adam GD: {hist_adam[-1]} | f(x,y) = {f(hist_adam[-1]):.10f}")

Final position after 100 iterations (target: [0.0, 0.0]):

Adam GD: [-0.00434583 -0.00434583] | f(x,y) = 0.0001133174


## 5. Implement the Newton method and BFGS